# 06. scenarios: {요인: 이동} 사전을 만드는 규칙 함수

`src/quanttool/analysis/scenarios.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| `market` (요인 격자: `objects[커브].tenors`, `.times`) | 시나리오 사전. 시장을 건드리지 않는다 (적용은 `market.scenario`) |

**객체 지도** (이 순서로 본다)

1. `Scenario` 타입
2. `parallel`
3. `from_loadings`
4. `combine`
5. `triangle_weight`
6. `bucket`

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from quanttool.config import load_config
from quanttool.market.quotes import read_quotes

cfg = load_config(ROOT / "configs")
quotes = read_quotes(FIX / "market_sample.txt")
q = quotes[ASOF]

from quanttool.analysis.scenarios import (
    Scenario,
    bucket,
    combine,
    from_loadings,
    parallel,
    triangle_weight,
)
from quanttool.market import build_market

market = build_market(ASOF, q, cfg)
Scenario

## `parallel(market, 커브, 크기)`: 격자 전체 같은 이동

In [ ]:
parallel(market, "CD91", 0.001)

## `from_loadings(커브, {만기: 로딩}, 크기)`: 크기 × 로딩 (PCA 모양)
아래 로딩은 설명용으로 만든 값이다. 실제 추정 로딩은 `configs/estimated/`에 출처와 함께 둔다.

In [ ]:
obj = market.objects["UST"]
loading = {
    t: -1 + 2 * time / max(obj.times.values()) for t, time in obj.times.items()
}  # 단기 −1 → 장기 +1
steep = from_loadings("UST", loading, 0.0005)
pd.Series(steep).round(6).head(8)

## `combine(s1, s2, …)`: 이동을 더한다 (여러 커브를 한 시나리오에)

In [ ]:
combine(parallel(market, "KOFR", 0.001), {"CD91/5Y": 0.0005, "KOFR/5Y": 0.0002})

## `triangle_weight` → `bucket`: key rate의 재료
이동 만기 j의 삼각형 가중치(양 끝 밖은 평탄). 모든 j의 가중치를 더하면 1, 그래서 bucket을 다 더하면 평행 이동.

In [ ]:
shift = ["2Y", "5Y", "10Y", "30Y"]
ref = obj.base.referenceDate()
from quanttool.market.curves import DC

st = [DC.yearFraction(ref, ref + ql.Period(t)) for t in shift]
w = pd.DataFrame(
    {s: {g: triangle_weight(st, j, tm) for g, tm in obj.times.items()} for j, s in enumerate(shift)}
)
w["합"] = w.sum(axis=1)
w.round(3)

In [ ]:
b5 = bucket(market, "UST", shift, 1, 1e-4)
total = combine(*(bucket(market, "UST", shift, j, 1e-4) for j in range(len(shift))))
print("5Y bucket:", {k: round(v * 1e4, 3) for k, v in b5.items()}, "(bp)")
print(
    "bucket 합 = 평행:",
    all(abs(total[k] - v) < 1e-15 for k, v in parallel(market, "UST", 1e-4).items()),
)

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `parallel` | 시장, 커브, 크기 | `{커브/만기: 크기}` | 평행 충격, 평행 DV01 |
| `from_loadings` | 커브, 로딩, 크기 | 시나리오 | PCA·스티프닝 |
| `combine` | 시나리오들 | 합친 시나리오 | 여러 충격 동시 |
| `bucket` | 시장, 커브, 이동 만기, j, 크기 | 시나리오 | key rate (risk) |

## 내 메모

- 